[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_09_Capstone_AutoResearcher.ipynb)

# 🎓 Lesson 9: Capstone Project — **AutoResearcher Agent**
### Build a Production-Ready AI Research Agent from Scratch

---

Welcome to the **final lesson** of your AI Engineering curriculum! You've spent 8 lessons
mastering the building blocks. Today you synthesize **every single skill** into one
cohesive, open-source-ready project: **AutoResearcher**.

> **AutoResearcher** is an autonomous agent that takes any topic, searches the web,
> reads articles, builds a private knowledge base, critiques its own work, and
> produces a professional research report — all without human intervention.

---

## 🗺️ Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                    ORCHESTRATOR                              │
│                                                             │
│  ┌─────────────────┐    ┌──────────────────────────────┐   │
│  │  RESEARCH AGENT  │    │       CRITIC AGENT           │   │
│  │  (ReAct Loop)    │───▶│  "Is the research good       │   │
│  │                 │    │   enough? Score it 1-10"      │   │
│  │  Tools:         │    └──────────────┬───────────────┘   │
│  │  • web_search   │                   │                    │
│  │  • fetch_url    │                   │ Verdict:           │
│  │  • save_note    │◀──────────────────┘ APPROVED or        │
│  │  • search_notes │                     NEEDS_MORE         │
│  └────────┬────────┘                                        │
│           │                                                 │
│           ▼                                                 │
│  ┌────────────────┐     ┌──────────────────────────────┐   │
│  │  ChromaDB      │     │    REPORT WRITER             │   │
│  │  Vector Store  │────▶│  "Generate final markdown    │   │
│  │  (RAG Memory)  │     │   report from all notes"     │   │
│  └────────────────┘     └──────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
```

---

## 📚 Skills From Every Lesson

| Lesson | Skill | Where It's Used |
|--------|-------|----------------|
| 1 — LLM Fundamentals | API calls, tokens, models | Every agent call |
| 2 — Prompt Engineering | System prompts, output formatting | Researcher + Critic + Reporter prompts |
| 3 — Tool Use | Tool definitions, tool loop | `web_search`, `fetch_url`, `save_note`, `search_notes` |
| 4 — ReAct Agent | Agent loop, self-correction | Core Research Agent |
| 5 — Memory | Short-term (context), long-term (DB) | ChromaDB persistence across runs |
| 6 — Multi-Agent | Orchestrator + subagents, critic loop | Orchestrator → Researcher → Critic → Reporter |
| 7 — RAG | Embeddings, vector search, retrieval | `search_notes` retrieves relevant past findings |
| 8 — Productionizing | Cost tracking, retries, logging | `@retry`, token counters, trace logs |

---

## 🎯 By the End of This Notebook

You will have a **working, runnable agent** that you can:
1. Run on any topic right now in Colab
2. Push to GitHub as an open-source project
3. Show as a portfolio piece demonstrating all AI engineering skills


---
## ⚙️ Part 1: Environment Setup

First, install all dependencies. This takes ~60 seconds on a fresh Colab runtime.


In [ ]:
# Install all required packages
!pip install anthropic chromadb duckduckgo-search beautifulsoup4 requests tenacity -q

print("✅ All packages installed!")


### 🔑 API Key Setup

1. In the left sidebar, click **🔑 Secrets** (key icon)
2. Add a secret named `ANTHROPIC_API_KEY` with your key
3. Toggle "Notebook access" to ON
4. Run the cell below


In [ ]:
import anthropic
import chromadb
import requests
import json
import uuid
import time
import logging
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
from IPython.display import Markdown, display

# ── API Key (from Colab Secrets) ──────────────────────────────────────────────
try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    import os
    API_KEY = os.getenv("ANTHROPIC_API_KEY", "your-key-here")

client = anthropic.Anthropic(api_key=API_KEY)

# ── Model choice (Haiku = fast + cheap for research loops) ───────────────────
# 💡 EXPERIMENT: Try "claude-3-5-sonnet-20241022" for higher-quality research
MODEL = "claude-3-5-haiku-20241022"

# ── Logging setup ─────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("autoresearcher")

print(f"✅ AutoResearcher ready | Model: {MODEL}")


---
## 🔧 Part 2: Tool Definitions (Lesson 3 Revisited)

Four tools power the research agent. Notice how each is a plain Python function —
the LLM decides *when* and *how* to call them.

### Tool 1: `web_search` — DuckDuckGo (no API key needed!)


In [ ]:
def web_search(query: str, max_results: int = 5) -> list[dict]:
    """Search the web using DuckDuckGo. No API key required."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
            return [
                {"title": r["title"], "url": r["href"], "snippet": r["body"][:300]}
                for r in results
            ]
    except Exception as e:
        # Graceful degradation if DuckDuckGo rate-limits
        return [{"error": str(e), "fallback": f"Search failed for: {query}"}]

# Quick test
results = web_search("transformer architecture LLM 2024", max_results=2)
for r in results:
    print(f"📰 {r.get('title', 'No title')}")
    print(f"   {r.get('snippet', r.get('error', ''))[:100]}...")
    print()


### Tool 2: `fetch_url` — Read full article content


In [ ]:
def fetch_url(url: str, max_chars: int = 3000) -> str:
    """Fetch and extract readable text content from a webpage."""
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (AutoResearcher/1.0; educational AI agent)"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Remove boilerplate elements
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()

        # Extract meaningful lines (>30 chars filters out nav crumbs, etc.)
        text = soup.get_text(separator="\n", strip=True)
        lines = [line.strip() for line in text.split("\n") if len(line.strip()) > 30]
        content = "\n".join(lines[:80])  # First 80 meaningful lines

        return content[:max_chars] if content else "[No readable content found]"

    except requests.Timeout:
        return f"[Timeout fetching {url}]"
    except Exception as e:
        return f"[Error: {e}]"

# 💡 EXPERIMENT: Try fetching a Wikipedia article URL
print(fetch_url("https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)")[:500])
print("\n... (truncated)")


### Tools 3 & 4: `save_note` + `search_notes` — ChromaDB RAG Memory (Lesson 7)

This is the heart of the agent's long-term memory. Every useful finding gets embedded
and stored. When researching, the agent first checks here before hitting the web.


In [ ]:
# ── Initialize ChromaDB (in-memory for this session) ─────────────────────────
chroma_client = chromadb.Client()

# Create collection — ChromaDB automatically handles embeddings!
try:
    collection = chroma_client.get_collection("research_notes")
    print("📂 Loaded existing research_notes collection")
except Exception:
    collection = chroma_client.create_collection(
        name="research_notes",
        metadata={"hnsw:space": "cosine"}  # Cosine similarity for semantic search
    )
    print("📂 Created fresh research_notes collection")


def save_note(title: str, content: str, topic: str) -> str:
    """Save a research finding to the vector store for later RAG retrieval."""
    doc_id = str(uuid.uuid4())[:8]
    collection.add(
        documents=[f"TITLE: {title}\n\n{content}"],
        metadatas=[{"title": title, "topic": topic}],
        ids=[doc_id]
    )
    return f"✅ Saved: '{title}' (id: {doc_id}) | Total notes: {collection.count()}"


def search_notes(query: str, n_results: int = 3) -> list[dict]:
    """Semantic search over saved notes (RAG retrieval). Use BEFORE web searching."""
    count = collection.count()
    if count == 0:
        return [{"message": "Knowledge base is empty — use web_search first"}]

    results = collection.query(
        query_texts=[query],
        n_results=min(n_results, count)
    )
    return [
        {
            "content": doc[:400],
            "metadata": meta,
            "relevance_rank": i + 1
        }
        for i, (doc, meta) in enumerate(
            zip(results["documents"][0], results["metadatas"][0])
        )
    ]


# Quick test
print(save_note(
    title="Test note",
    content="This is a test to verify ChromaDB is working correctly.",
    topic="test"
))
print("\nSearch result:", search_notes("test chromadb")[0]["content"][:80])

# Clean up test note
collection.delete(where={"topic": "test"})
print("🧹 Test note cleaned up | Notes remaining:", collection.count())


---
## 📋 Part 3: Tool Registry (The Bridge Between Code & LLM)

The tool registry is the schema the LLM reads to understand *what it can do*.
It's the exact same pattern from Lesson 3 — JSON Schema definitions.


In [ ]:
# ── Tool Schema Definitions (what Claude sees) ───────────────────────────────
TOOLS = [
    {
        "name": "web_search",
        "description": (
            "Search the web for information about a topic. Returns titles, URLs, and snippets. "
            "Use this to discover sources. Follow with fetch_url to get full content."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query to use"},
                "max_results": {"type": "integer", "description": "Max results to return (default: 5)"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "fetch_url",
        "description": (
            "Fetch and read the full text content of a webpage. "
            "Use after web_search to read promising articles in depth."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "url": {"type": "string", "description": "The full URL to fetch and read"}
            },
            "required": ["url"]
        }
    },
    {
        "name": "save_note",
        "description": (
            "Save an important research finding to the knowledge base. "
            "Save key facts, statistics, definitions, and insights. "
            "Be selective — only save genuinely useful information."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string", "description": "Short descriptive title for the finding"},
                "content": {"type": "string", "description": "The research content to save (2-5 sentences)"},
                "topic": {"type": "string", "description": "The research topic this note belongs to"}
            },
            "required": ["title", "content", "topic"]
        }
    },
    {
        "name": "search_notes",
        "description": (
            "Search previously saved research notes using semantic similarity. "
            "ALWAYS call this first before searching the web to avoid duplicate work. "
            "This is your long-term memory — check it!"
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "What to search for in your notes"},
                "n_results": {"type": "integer", "description": "Number of notes to retrieve (default: 3)"}
            },
            "required": ["query"]
        }
    }
]


def execute_tool(name: str, inputs: dict) -> str:
    """Route a tool call to the correct Python function."""
    try:
        if name == "web_search":
            result = web_search(inputs["query"], inputs.get("max_results", 5))
        elif name == "fetch_url":
            result = fetch_url(inputs["url"])
        elif name == "save_note":
            result = save_note(inputs["title"], inputs["content"], inputs["topic"])
        elif name == "search_notes":
            result = search_notes(inputs["query"], inputs.get("n_results", 3))
        else:
            result = f"⚠️ Unknown tool: {name}"

        # Convert to string (LLM expects string results)
        return json.dumps(result, indent=2) if isinstance(result, (list, dict)) else str(result)

    except Exception as e:
        return f"[Tool error in {name}: {e}]"


print(f"✅ Tool registry ready | {len(TOOLS)} tools: {[t['name'] for t in TOOLS]}")


---
## 🤖 Part 4: The Research Agent — ReAct Loop (Lesson 4 Revisited)

This is the core of the system. The agent loops: **think → act (call a tool) → observe → repeat**
until it decides it has gathered enough information.

Key improvements over Lesson 4:
- **Retry logic** with exponential backoff (`tenacity`) from Lesson 8
- **Token + cost tracking** from Lesson 8
- **Trace logging** for observability
- **RAG first** — checks memory before hitting the web


In [ ]:
# ── Researcher System Prompt (Lesson 2: Prompt Engineering) ──────────────────
RESEARCHER_SYSTEM_PROMPT = """You are AutoResearcher, a meticulous AI research agent.

Your mission: Research a given topic thoroughly by gathering high-quality, specific information.

## Research Protocol (follow in order):
1. **Check memory first**: Call search_notes to see what you already know
2. **Web search**: Use web_search with specific, targeted queries
3. **Deep read**: Call fetch_url on the most promising 2-3 results
4. **Save insights**: Call save_note for each important finding (key facts, stats, definitions, quotes)
5. **Iterate**: Run 2-3 different search angles to get diverse coverage
6. **Stop when ready**: Once you have 4+ quality notes saved, output [RESEARCH_COMPLETE]

## Rules:
- Save notes that are SPECIFIC (include numbers, dates, names) not generic
- Use varied search queries — don't repeat the same search
- If a page fails to load, try the next result
- Each save_note should capture ONE distinct insight (not a wall of text)

When you have gathered sufficient information, write: [RESEARCH_COMPLETE]
Do NOT write a report — that's handled separately.
"""


# ── LLM call with retry (Lesson 8: Productionizing) ─────────────────────────
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=15),
    retry=retry_if_exception_type((anthropic.APIConnectionError, anthropic.RateLimitError))
)
def call_claude(messages: list, system: str, tools: list = None) -> anthropic.types.Message:
    """LLM call with automatic retry on transient failures."""
    kwargs = {
        "model": MODEL,
        "max_tokens": 2048,
        "system": system,
        "messages": messages
    }
    if tools:
        kwargs["tools"] = tools
    return client.messages.create(**kwargs)


# ── Core ReAct Agent Loop ────────────────────────────────────────────────────
def research_agent(topic: str, max_iterations: int = 12) -> tuple:
    """
    ReAct research loop: Reason → Act (tool call) → Observe → repeat.

    Returns:
        messages: Full conversation history
        stats: Token usage + cost + tool call trace
    """
    messages = [{"role": "user", "content": f"Research this topic thoroughly: **{topic}**"}]

    # Observability: stats dict (Lesson 8)
    stats = {
        "total_input_tokens": 0,
        "total_output_tokens": 0,
        "tool_calls": 0,
        "iterations": 0,
        "trace": []  # What tools were called and when
    }

    print(f"\n🔍 Research Agent starting on: '{topic}'")
    print("─" * 55)

    for i in range(max_iterations):
        stats["iterations"] += 1

        # ── Call the LLM ──────────────────────────────────────────────────────
        response = call_claude(messages, RESEARCHER_SYSTEM_PROMPT, TOOLS)

        # Track tokens (Lesson 8: cost optimization)
        stats["total_input_tokens"] += response.usage.input_tokens
        stats["total_output_tokens"] += response.usage.output_tokens

        # ── Check if agent decided to stop ────────────────────────────────────
        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text

            if "[RESEARCH_COMPLETE]" in final_text:
                print(f"\n✅ Research complete after {i+1} iterations")
                stats["trace"].append({"action": "COMPLETE", "iteration": i+1})
            else:
                print(f"\n⚠️  Agent stopped unexpectedly at iteration {i+1}")
                print(f"   Last message: {final_text[:100]}...")
            break

        # ── Process tool calls ────────────────────────────────────────────────
        if response.stop_reason == "tool_use":
            # Add assistant turn to history
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    stats["tool_calls"] += 1
                    tool_name = block.name
                    tool_input = block.input

                    # Show progress
                    preview = str(list(tool_input.values())[0])[:50] if tool_input else ""
                    print(f"  [{i+1:02d}] 🔧 {tool_name}({preview}...)")

                    # Execute the tool
                    result = execute_tool(tool_name, tool_input)

                    # Log to trace
                    stats["trace"].append({
                        "iteration": i + 1,
                        "tool": tool_name,
                        "input_preview": preview,
                        "result_length": len(result)
                    })

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result[:2500]  # Cap to manage context window
                    })

            # Feed tool results back to agent
            messages.append({"role": "user", "content": tool_results})

    # ── Cost calculation ──────────────────────────────────────────────────────
    # claude-3-5-haiku-20241022 pricing
    INPUT_COST_PER_1M  = 0.80   # $0.80 per 1M input tokens
    OUTPUT_COST_PER_1M = 4.00   # $4.00 per 1M output tokens

    stats["estimated_cost_usd"] = round(
        (stats["total_input_tokens"]  / 1_000_000 * INPUT_COST_PER_1M) +
        (stats["total_output_tokens"] / 1_000_000 * OUTPUT_COST_PER_1M),
        5
    )

    print(f"\n📊 Agent Stats:")
    print(f"   Iterations : {stats['iterations']}")
    print(f"   Tool calls : {stats['tool_calls']}")
    print(f"   Tokens     : {stats['total_input_tokens']:,} in / {stats['total_output_tokens']:,} out")
    print(f"   Est. cost  : ${stats['estimated_cost_usd']:.4f}")
    print(f"   Notes saved: {collection.count()}")

    return messages, stats


print("✅ Research Agent defined")


---
## 🧐 Part 5: The Critic Agent (Lesson 6: Multi-Agent Systems)

The critic is a separate subagent with a **different role and system prompt**.
It reads the research notes and scores the quality — just like a human peer reviewer.

This pattern (Orchestrator → Worker → Critic) is the most common multi-agent
architecture in production AI systems.


In [ ]:
CRITIC_SYSTEM_PROMPT = """You are a rigorous research quality evaluator.
Your job: assess whether research notes on a topic are sufficient to write a high-quality report.

Evaluate based on:
- DEPTH: Are findings specific (numbers, names, dates) or vague?
- BREADTH: Are multiple angles/perspectives covered?
- CREDIBILITY: Are there references to credible sources?
- COMPLETENESS: Would a reader understand the topic well after reading this?

Respond with ONLY valid JSON — no markdown, no explanation outside the JSON:
{
  "quality_score": <integer 1-10>,
  "coverage_strengths": ["<strength 1>", "<strength 2>"],
  "gaps": ["<gap 1>", "<gap 2>"],
  "verdict": "<APPROVED or NEEDS_MORE_RESEARCH>",
  "guidance_for_next_round": "<specific suggestion if NEEDS_MORE_RESEARCH>"
}

APPROVED = score >= 7 AND at least 3 substantive notes exist.
NEEDS_MORE_RESEARCH = score < 7 OR fewer than 3 quality notes.
"""


def critic_agent(topic: str) -> dict:
    """
    Evaluates research quality by reading all stored notes.
    Returns a structured critique dict.
    """
    # Retrieve all notes about this topic via RAG
    stored = search_notes(topic, n_results=15)
    note_count = len([n for n in stored if "message" not in n])

    if note_count == 0:
        return {
            "quality_score": 0,
            "verdict": "NEEDS_MORE_RESEARCH",
            "gaps": ["No research notes found at all"],
            "guidance_for_next_round": "Run the research agent first."
        }

    notes_text = "\n\n".join([
        f"[Note {i+1}] {n.get('metadata', {}).get('title', 'Untitled')}\n{n['content'][:300]}"
        for i, n in enumerate(stored) if "message" not in n
    ])

    critique_prompt = f"""Topic: {topic}

Number of notes in knowledge base: {note_count}

Research notes collected:
{notes_text}

Evaluate the research quality for this topic."""

    response = call_claude(
        messages=[{"role": "user", "content": critique_prompt}],
        system=CRITIC_SYSTEM_PROMPT
    )

    text = response.content[0].text

    # Parse JSON from response
    try:
        start = text.find("{")
        end = text.rfind("}") + 1
        if start >= 0:
            return json.loads(text[start:end])
    except json.JSONDecodeError:
        pass

    # Fallback if JSON parsing fails
    return {
        "quality_score": 5,
        "verdict": "NEEDS_MORE_RESEARCH",
        "gaps": ["Could not parse critique response"],
        "guidance_for_next_round": "Retry research with different angles."
    }


print("✅ Critic Agent defined")


---
## 📝 Part 6: Report Writer Agent

The report writer is the final subagent. It uses **RAG** to retrieve all stored notes
and synthesizes them into a professional markdown report — the deliverable.


In [ ]:
REPORT_WRITER_SYSTEM_PROMPT = """You are a professional research writer and analyst.
Your task: synthesize research notes into a polished, insightful markdown report.

Report structure:
# [Topic] — Research Report

## Executive Summary
(2-3 sentences capturing the most important takeaways)

## Key Findings
(4-6 specific, substantiated bullet points with numbers/facts where available)

## Deep Dive: [Major Theme 1]
(Paragraph covering the first major theme in depth)

## Deep Dive: [Major Theme 2]
(Paragraph covering the second major theme in depth)

## Conclusion & Implications
(What this means, what's significant, what to watch)

## Open Questions
(2-3 questions worth exploring further)

---
*Report generated by AutoResearcher | [date]*

Rules:
- Be SPECIFIC: use numbers, names, dates from the research
- Do NOT invent facts not present in the notes
- Write for a technical audience who wants substance, not fluff
- Use proper markdown formatting
"""


def generate_report(topic: str) -> str:
    """Generate a final report by RAG-retrieving all notes and synthesizing them."""
    # RAG: retrieve comprehensive notes
    notes = search_notes(topic, n_results=20)
    valid_notes = [n for n in notes if "message" not in n]

    if not valid_notes:
        return f"# {topic}\n\n⚠️ No research notes found. Run the research agent first."

    notes_text = "\n\n".join([
        f"### {n.get('metadata', {}).get('title', f'Note {i+1}')}\n{n['content']}"
        for i, n in enumerate(valid_notes)
    ])

    from datetime import datetime
    date_str = datetime.now().strftime("%Y-%m-%d")

    response = call_claude(
        messages=[{
            "role": "user",
            "content": (
                f"Write a comprehensive research report on: **{topic}**\n\n"
                f"Research notes gathered ({len(valid_notes)} notes):\n\n{notes_text}\n\n"
                f"Today's date: {date_str}"
            )
        }],
        system=REPORT_WRITER_SYSTEM_PROMPT
    )

    return response.content[0].text


print("✅ Report Writer defined")


---
## 🎼 Part 7: The Orchestrator — Tying It All Together (Lesson 6)

The orchestrator is the conductor. It:
1. Runs the **Research Agent** to gather information
2. Calls the **Critic** to evaluate quality
3. If quality is low, runs another research round with targeted guidance
4. Finally calls the **Report Writer** to produce the deliverable

This is the **full multi-agent pipeline** from Lesson 6 — now with real tools and real RAG.


In [ ]:
def auto_researcher(
    topic: str,
    max_research_rounds: int = 2,
    verbose: bool = True
) -> str:
    """
    AutoResearcher Orchestrator — the main entry point.

    Runs the complete pipeline:
      Research Agent → Critic → (optional: more research) → Report Writer

    Args:
        topic: What to research
        max_research_rounds: Max times to loop research+critique (default: 2)
        verbose: Print progress (default: True)

    Returns:
        str: Final markdown report
    """

    print("\n" + "═" * 60)
    print("🤖  A U T O R E S E A R C H E R")
    print("═" * 60)
    print(f"📋  Topic  : {topic}")
    print(f"🔄  Max rounds: {max_research_rounds}")
    print("═" * 60)

    total_cost = 0.0
    critique_history = []

    for round_num in range(1, max_research_rounds + 1):

        # ── Phase 1: Research ─────────────────────────────────────────────────
        print(f"\n📚 ROUND {round_num}/{max_research_rounds} — Research Phase")

        # If we have a critique from last round, guide the agent to fill gaps
        research_topic = topic
        if critique_history:
            last_critique = critique_history[-1]
            gaps = ", ".join(last_critique.get("gaps", [])[:2])
            guidance = last_critique.get("guidance_for_next_round", "")
            research_topic = (
                f"{topic}. "
                f"Focus specifically on these gaps: {gaps}. "
                f"{guidance}"
            )
            print(f"  📌 Targeting gaps: {gaps[:80]}...")

        messages, stats = research_agent(research_topic)
        total_cost += stats["estimated_cost_usd"]

        # ── Phase 2: Critique ─────────────────────────────────────────────────
        print(f"\n🧐 ROUND {round_num}/{max_research_rounds} — Quality Evaluation")
        critique = critic_agent(topic)
        critique_history.append(critique)

        score   = critique.get("quality_score", 0)
        verdict = critique.get("verdict", "NEEDS_MORE_RESEARCH")
        gaps    = critique.get("gaps", [])

        print(f"  📊 Quality Score : {score}/10")
        print(f"  🏷️  Verdict       : {verdict}")
        if gaps:
            print(f"  📝 Gaps          : {'; '.join(gaps[:2])}")

        # ── Decide: stop or continue ──────────────────────────────────────────
        if verdict == "APPROVED" or round_num == max_research_rounds:
            if verdict == "APPROVED":
                print(f"\n  ✅ Research approved! Moving to report generation.")
            else:
                print(f"\n  ⚠️  Max rounds reached. Generating report with available research.")
            break
        else:
            print(f"\n  🔄 Quality below threshold — running another research round...")

    # ── Phase 3: Generate Report ──────────────────────────────────────────────
    print(f"\n📝 FINAL PHASE — Generating Report")
    report = generate_report(topic)

    # ── Final Summary ─────────────────────────────────────────────────────────
    print(f"\n{'═' * 60}")
    print(f"✅  COMPLETE")
    print(f"   Research rounds : {round_num}")
    print(f"   Notes in KB     : {collection.count()}")
    print(f"   Total cost      : ${total_cost:.4f}")
    print(f"   Final score     : {critique_history[-1].get('quality_score', 'N/A')}/10")
    print(f"{'═' * 60}")

    return report


print("✅ Orchestrator ready — AutoResearcher is fully assembled!")


---
## 🚀 Part 8: Run AutoResearcher!

Time to launch the full agent. Change the `RESEARCH_TOPIC` variable below to
research anything you're curious about.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 💡 EXPERIMENT: Change this topic to anything you want to research!
# ─────────────────────────────────────────────────────────────────────────────
RESEARCH_TOPIC = "How transformer attention mechanisms work in large language models"

# Other ideas to try:
# RESEARCH_TOPIC = "The business impact of AI agents in 2024"
# RESEARCH_TOPIC = "Retrieval augmented generation vs fine-tuning tradeoffs"
# RESEARCH_TOPIC = "Vector databases: Pinecone vs Weaviate vs ChromaDB comparison"
# RESEARCH_TOPIC = "Model Context Protocol (MCP) and its role in AI agents"

# ─────────────────────────────────────────────────────────────────────────────
# Reset knowledge base for a fresh run
# (remove this line if you want to build on previous research)
# ─────────────────────────────────────────────────────────────────────────────
try:
    chroma_client.delete_collection("research_notes")
except:
    pass
collection = chroma_client.create_collection(
    name="research_notes",
    metadata={"hnsw:space": "cosine"}
)

# ─────────────────────────────────────────────────────────────────────────────
# 🚀 LAUNCH THE AGENT
# ─────────────────────────────────────────────────────────────────────────────
report = auto_researcher(
    topic=RESEARCH_TOPIC,
    max_research_rounds=2,  # 💡 EXPERIMENT: Try 3 for deeper research
    verbose=True
)

# ─────────────────────────────────────────────────────────────────────────────
# Display the report beautifully
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("📄  FINAL REPORT")
print("═" * 60 + "\n")
display(Markdown(report))


---
## 💾 Part 9: Save Your Report

Save the generated report to a markdown file — ready to publish on GitHub or your blog.


In [ ]:
from datetime import datetime

# Save report to file
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
safe_topic = RESEARCH_TOPIC.replace(" ", "_").replace("/", "-")[:40]
filename = f"report_{safe_topic}_{timestamp}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write(f"# Research Report: {RESEARCH_TOPIC}\n")
    f.write(f"*Generated by AutoResearcher on {datetime.now().strftime('%Y-%m-%d %H:%M')}*\n\n")
    f.write(report)

print(f"✅ Report saved to: {filename}")
print(f"   File size: {len(report):,} chars")
print("\n📥 To download from Colab: Files panel (left sidebar) → right-click → Download")


---
## 🔎 Part 10: Inspect the Knowledge Base

See exactly what your agent saved to ChromaDB during the research run.
This is your agent's *memory* — what it learned and retained.


In [ ]:
# Inspect all saved notes
all_notes = collection.get()

print(f"📚 Knowledge Base Contents ({len(all_notes['ids'])} notes):")
print("=" * 60)

for i, (doc_id, doc, meta) in enumerate(
    zip(all_notes["ids"], all_notes["documents"], all_notes["metadatas"])
):
    print(f"\n[{i+1}] 📌 {meta.get('title', 'Untitled')}")
    print(f"    ID: {doc_id} | Topic: {meta.get('topic', 'unknown')}")
    print(f"    {doc[7:200]}...")  # Skip "TITLE: " prefix

print("\n" + "=" * 60)
print(f"Total: {len(all_notes['ids'])} notes stored in ChromaDB")
print("\n💡 EXPERIMENT: Try search_notes() with a specific query:")
print("   results = search_notes('attention mechanism')")
print("   print(results[0]['content'])")


---
## 🌐 Part 11: Going Open Source — Your Portfolio Project

You've built a working AI agent. Now let's make it GitHub-ready.

### Recommended Repository Structure

```
auto-researcher/
├── README.md                    ← Project description + demo GIF
├── auto_researcher.py           ← Core agent (extracted from this notebook)
├── notebooks/
│   └── Lesson_09_Capstone_AutoResearcher.ipynb   ← This notebook
├── reports/
│   └── sample_report_transformers.md              ← Example output
├── requirements.txt
└── .env.example                 ← Template: ANTHROPIC_API_KEY=your-key-here
```

### README Template (copy and customize)

```markdown
# 🤖 AutoResearcher

An autonomous AI research agent that searches the web, builds a knowledge base,
and generates professional research reports — without human intervention.

Built as a capstone project for the [AI Engineering Curriculum](link).

## What It Does

1. Takes any research topic
2. Searches DuckDuckGo for relevant sources
3. Reads and extracts content from web pages
4. Stores findings in ChromaDB (vector database)
5. Self-evaluates research quality with a critic agent
6. Generates a professional markdown report

## Tech Stack

- **LLM**: Anthropic Claude (claude-3-5-haiku)
- **Agent Pattern**: ReAct (Reason + Act) loop
- **Vector DB**: ChromaDB for semantic search
- **Web Search**: DuckDuckGo (no API key needed)
- **Architecture**: Multi-agent (Researcher + Critic + Reporter)

## Quick Start

\`\`\`bash
pip install anthropic chromadb duckduckgo-search beautifulsoup4 requests tenacity
export ANTHROPIC_API_KEY=your-key-here
python auto_researcher.py --topic "transformer attention mechanisms"
\`\`\`

## Architecture

[paste ASCII diagram from notebook here]

## Skills Demonstrated

✅ LLM API integration  ✅ Prompt engineering  ✅ Tool use / function calling
✅ ReAct agent loop     ✅ Long-term memory    ✅ Multi-agent systems
✅ RAG pipeline         ✅ Production patterns (retries, cost tracking, logging)
```

### Next Steps to Polish Your Open-Source Project

1. **Extract to `.py` file** — copy each function from this notebook into `auto_researcher.py`
2. **Add CLI** — use `argparse` so users can run `python auto_researcher.py --topic "AI trends"`
3. **Persistent storage** — use `chromadb.PersistentClient("/path/to/db")` to keep notes across runs
4. **GitHub Actions** — add a daily scheduled research run that commits reports to the repo
5. **Demo GIF** — record a terminal session showing the agent at work


---
## 🎓 Curriculum Complete — What's Next?

Congratulations! You've completed the full 9-lesson AI Engineering curriculum.

### What You've Built & Mastered

| ✅ | Skill | Proof |
|----|-------|-------|
| ✅ | LLM API & streaming | Lesson 1 |
| ✅ | Prompt Engineering | Lesson 2 |
| ✅ | Tool Use | Lesson 3 |
| ✅ | ReAct Agent | Lesson 4 |
| ✅ | Agent Memory | Lesson 5 |
| ✅ | Multi-Agent Systems | Lesson 6 |
| ✅ | RAG Pipeline | Lesson 7 |
| ✅ | Productionizing AI | Lesson 8 |
| ✅ | **Capstone Agent** | **This notebook** |

---

### 🚀 Your Next Challenges (post-curriculum exploration)

**Level Up: Specialization**
- **Fine-tuning**: Train a model on your own data with Unsloth or HuggingFace PEFT
- **Multimodal agents**: Add vision capabilities (Claude's vision API)
- **Voice agents**: Add speech-to-text + TTS for voice interaction
- **Computer-use agents**: Control a browser/desktop programmatically

**Production Hardening**
- **Streaming UI**: Add a real-time web UI with FastAPI + WebSockets
- **Auth & multi-tenancy**: Per-user knowledge bases in ChromaDB
- **Eval framework**: Build automated evals for your agent's output quality
- **Cost budgets**: Hard token limits + user-facing cost estimates

**Open Source to Portfolio**
- Submit AutoResearcher to an AI agent directory (e.g., awesome-ai-agents)
- Write a blog post explaining your architecture (dev.to, Hashnode, Substack)
- Present it at a local meetup or AI community

---

### 💬 Keep the Conversation Going

Your scheduled daily sessions will continue — ask questions, share what you've built,
and we'll evolve the curriculum based on what you want to explore next.

**You are now an AI Engineer. Go build things. 🛠️**

---
*AutoResearcher Capstone | Lesson 9 of 9 | AI Engineering Curriculum*
